In [1]:
import os
import textwrap
import pandas as pd

# Procura os arquivos em ./dados (padrão do experimento) ou no diretório atual
def _achar(nome):
    for base in ('dados', '.', '..'):
        c = os.path.join(base, nome)
        if os.path.isfile(c):
            return c
    raise FileNotFoundError(nome)

ARQ_INTEGRAS = _achar('integras_experimento_summa.parquet')
ARQ_DIVISAO = _achar('divisao_Qwen235b_Qwen7b.csv')
# Saída esperada (JSON em texto, coluna 'resposta') gerada pelo Qwen 235b; a coluna 'pasta' é ignorada
ARQ_SAIDA = _achar(os.path.join('saida', 'saida_or_235b.parquet'))
for nome, arq in [('íntegras', ARQ_INTEGRAS), ('divisão', ARQ_DIVISAO), ('saída', ARQ_SAIDA)]:
    print(f'{nome:<9}: {arq}')

CHAVE = 'seq_documento_acordao'
COLS_DIVISAO = ['dificuldade', 'dificuldade_int', 'alvo', 'fold', 'token_total', 'token_output']

# ── Fonte primária: df_integras (uma linha por acórdão). Tudo o mais é ligado a ela por CHAVE;
#    registros de outras fontes sem correspondência nas íntegras são ignorados (left join).
FOLD_MAX = 10  # folds 1..10 compõem o experimento (11 e 12 são extrações posteriores)
df_integras_bruto = pd.read_parquet(ARQ_INTEGRAS)
# filtra pelo 'fold' das íntegras (vira 'fold_integras' após o join com a divisão)
df_integras = df_integras_bruto[df_integras_bruto['fold'] <= FOLD_MAX].reset_index(drop=True)
print(f'\níntegras: {len(df_integras_bruto):,} carregadas → {len(df_integras):,} com fold <= {FOLD_MAX}')
assert df_integras[CHAVE].is_unique, 'chave duplicada nas íntegras'
chaves_integras = set(df_integras[CHAVE])

def _normalizar(d, col_chave):
    """Converte a chave para int64, descarta chaves inválidas/fora das íntegras e duplicatas."""
    d = d.assign(**{CHAVE: pd.to_numeric(d[col_chave], errors='coerce')})
    d = d.dropna(subset=[CHAVE]).astype({CHAVE: 'int64'})
    if col_chave != CHAVE:
        d = d.drop(columns=[col_chave])
    fora = (~d[CHAVE].isin(chaves_integras)).sum()
    d = d[d[CHAVE].isin(chaves_integras)].drop_duplicates(subset=CHAVE, keep='last')
    return d, fora

df_div, fora_div = _normalizar(pd.read_csv(ARQ_DIVISAO, usecols=['id'] + COLS_DIVISAO), 'id')

df_saida = pd.read_parquet(ARQ_SAIDA, columns=['chave', 'resposta']).rename(columns={'resposta': 'saida'})
df_saida = df_saida[df_saida['saida'].fillna('').str.strip() != '']   # só respostas preenchidas
df_saida, fora_saida = _normalizar(df_saida, 'chave')

# 'fold' das íntegras é mantido como 'fold_integras'; o 'fold' da divisão prevalece
df = (df_integras
      .merge(df_div, on=CHAVE, how='left', suffixes=('_integras', ''), validate='one_to_one')
      .merge(df_saida, on=CHAVE, how='left', validate='one_to_one'))

for col in ['dificuldade_int', 'fold', 'token_total', 'token_output']:
    df[col] = df[col].astype('Int64')

# token_total = entrada + saída; deriva-se a contagem de tokens de entrada (prompt + íntegra)
df['token_entrada'] = df['token_total'] - df['token_output']

assert len(df) == len(df_integras), 'o join alterou o número de linhas das íntegras'

print(f'\níntegras (base): {len(df_integras):,} linhas')
print(f'divisão        : {len(df_div):,} ligadas | {fora_div:,} ignoradas (fora das íntegras) | '
      f'{df["alvo"].isna().sum():,} íntegras sem divisão')
print(f'saída          : {len(df_saida):,} ligadas | {fora_saida:,} ignoradas (fora das íntegras) | '
      f'{df["saida"].isna().sum():,} íntegras sem saída')
df.head()
del df_saida
del df_div

íntegras : dados/integras_experimento_summa.parquet
divisão  : dados/divisao_Qwen235b_Qwen7b.csv
saída    : ./saida/saida_or_235b.parquet

íntegras: 22,000 carregadas → 20,000 com fold <= 10

íntegras (base): 20,000 linhas
divisão        : 19,712 ligadas | 2,443 ignoradas (fora das íntegras) | 288 íntegras sem divisão
saída          : 19,812 ligadas | 2,457 ignoradas (fora das íntegras) | 188 íntegras sem saída


## Divisão treino / validação / teste
Contagem sobre `df` (base = `df_integras`); íntegras sem correspondência na divisão aparecem como `(sem divisão)`.

In [2]:
cont = df['alvo'].fillna('(sem divisão)').value_counts()
print(f'Total de instâncias (= df_integras): {len(df):,}\n')
print(f'{"alvo":<15}{"qtd":>10}{"%":>8}{"com saída":>12}')
print('-' * 45)
for k, v in cont.items():
    m = df['alvo'].fillna('(sem divisão)') == k
    print(f'{str(k):<15}{v:>10,}{100 * v / len(df):>7.1f}%{df.loc[m, "saida"].notna().sum():>12,}')
print('-' * 45)
print(f'{"total":<15}{cont.sum():>10,}{100.0:>7.1f}%{df["saida"].notna().sum():>12,}')
assert cont.sum() == len(df_integras)

Total de instâncias (= df_integras): 20,000

alvo                  qtd       %   com saída
---------------------------------------------
treino             13,788   68.9%      13,788
teste               3,948   19.7%       3,948
validacao           1,976    9.9%       1,976
(sem divisão)         288    1.4%         100
---------------------------------------------
total              20,000  100.0%      19,812


## Lista de campos

In [3]:
def _resumo_valor(v, n=45):
    s = ' '.join(str(v).split())
    return s if len(s) <= n else s[:n - 1] + '…'

def listar_campos(d):
    print(f'{len(d):,} linhas × {d.shape[1]} campos\n')
    print(f'{"#":>3}  {"campo":<28}{"tipo":<14}{"nulos":>9}{"únicos":>10}  exemplo')
    print('-' * 115)
    for i, c in enumerate(d.columns, 1):
        s = d[c]
        try:
            nunq = s.nunique()
        except TypeError:  # colunas com listas/dicts
            nunq = s.astype(str).nunique()
        ex = s.dropna()
        ex = _resumo_valor(ex.iloc[0]) if len(ex) else '—'
        print(f'{i:>3}  {c:<28}{str(s.dtype):<14}{s.isna().sum():>9,}{nunq:>10,}  {ex}')

listar_campos(df)

20,000 linhas × 18 campos

  #  campo                       tipo              nulos    únicos  exemplo
-------------------------------------------------------------------------------------------------------------------
  1  sg_classe                   object                0         5  AREsp
  2  pasta                       object                0        10  ./saidas/<<tipo>>_01
  3  fold_integras               int64                 0        10  1
  4  dt_publicacao               object                0       349  2024-12-17
  5  sg_ramo_direito             object                0         5  TR
  6  num_registro                int64                 0    18,998  202301681297
  7  ano                         int64                 0         4  2024
  8  seq_documento_acordao       int64                 0    20,000  286006309
  9  num_ministro                int64                 0        20  1187
 10  integra                     object                0    19,813  ACÓRDÃO<br>Vistos e relat

## Contagem de tokens
`token_total` (entrada + saída), `token_entrada` (derivado) e `token_output` (JSON gerado pelo modelo), por conjunto (`alvo`) e por dificuldade.

In [ ]:
COLS_TOKENS = ['token_entrada', 'token_output', 'token_total']
ROTULOS = {'token_entrada': 'Entrada (prompt + íntegra)', 'token_output': 'Saída (JSON gerado)', 'token_total': 'Total'}


def tabela_tokens(d, titulo):
    print(f'\n▌ {titulo}  ({len(d):,} instâncias)')
    print(f'{"métrica":<28}{"média":>10}{"mediana":>10}{"desvio":>10}{"mín":>9}{"p90":>9}{"p99":>9}{"máx":>9}{"soma":>15}')
    print('─' * 109)
    for c in COLS_TOKENS:
        s = d[c].dropna()
        print(f'{ROTULOS[c]:<28}{s.mean():>10,.0f}{s.median():>10,.0f}{s.std():>10,.0f}{s.min():>9,.0f}'
              f'{s.quantile(.9):>9,.0f}{s.quantile(.99):>9,.0f}{s.max():>9,.0f}{s.sum():>15,.0f}')


df_tok = df[df['alvo'].notna()]  # tokens vêm da divisão: só íntegras ligadas a ela

print('█' * 109)
print(f'  CONTAGEM DE TOKENS — {len(df_tok):,} de {len(df):,} íntegras')
print('█' * 109)
tabela_tokens(df_tok, 'Todos os conjuntos')
for alvo in ['treino', 'validacao', 'teste']:
    tabela_tokens(df_tok[df_tok['alvo'] == alvo], f'alvo = {alvo}')

# Uma linha por nível de dificuldade (detalhes visuais na seção de gráficos abaixo)
print('\n▌ Tokens por dificuldade')
res = df_tok.groupby('dificuldade_int').agg(
    dificuldade=('dificuldade', 'first'), instancias=('token_total', 'size'),
    entrada_mediana=('token_entrada', 'median'), saida_mediana=('token_output', 'median'),
    total_medio=('token_total', 'mean'), total_p99=('token_total', lambda x: x.quantile(.99)),
    total_max=('token_total', 'max'))
print(res.to_string(float_format=lambda x: f'{x:,.0f}', formatters={'instancias': '{:,}'.format, 'total_max': '{:,}'.format}))

# Resumo comparativo compacto: média de tokens por alvo × dificuldade
print('\n▌ Média de token_total por alvo × dificuldade')
piv = df_tok.pivot_table(index='dificuldade_int', columns='alvo', values='token_total', aggfunc='mean').round(0)
piv.index = [f"{i} ({df_tok.loc[df_tok['dificuldade_int'] == i, 'dificuldade'].iloc[0]})" for i in piv.index]
print(piv.to_string(float_format=lambda x: f'{x:,.0f}'))

## Análise exploratória (gráficos)
Base: íntegras ligadas à divisão (`df_eda`), que são as que têm contagem de tokens. Cores fixas por conjunto (treino = azul, validação = laranja, teste = verde) e rampa azul clara→escura para `dificuldade_int` 1→10.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# ── Paleta: categórica em ordem fixa (alvo) e rampa sequencial azul (dificuldade 1..10)
COR = {'azul': '#2a78d6', 'laranja': '#eb6834', 'aqua': '#1baf7a',
       'texto': '#0b0b0b', 'texto2': '#52514e', 'grade': '#e4e3df', 'fundo': '#fcfcfb'}
ORDEM_ALVO = ['treino', 'validacao', 'teste']
COR_ALVO = dict(zip(ORDEM_ALVO, [COR['azul'], COR['laranja'], COR['aqua']]))
RAMPA_DIF = ['#86b6ef', '#6da7ec', '#5598e7', '#3987e5', '#2a78d6',
             '#256abf', '#1c5cab', '#184f95', '#104281', '#0d366b']
CMAP_AZUL = LinearSegmentedColormap.from_list('azul', ['#cde2fb', '#86b6ef', '#2a78d6', '#184f95', '#0d366b'])

plt.rcParams.update({
    'figure.facecolor': COR['fundo'], 'axes.facecolor': COR['fundo'], 'savefig.facecolor': COR['fundo'],
    'figure.dpi': 110, 'font.size': 9.5,
    'axes.edgecolor': COR['grade'], 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': COR['grade'], 'grid.linewidth': 0.7, 'axes.axisbelow': True,
    'axes.titlesize': 10.5, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.titlecolor': COR['texto'], 'axes.labelcolor': COR['texto2'],
    'xtick.color': COR['texto2'], 'ytick.color': COR['texto2'],
    'legend.frameon': False, 'figure.titlesize': 12.5, 'figure.titleweight': 'semibold',
})

FMT_MIL = mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
FMT_K = mticker.FuncFormatter(lambda x, _: f'{x / 1000:g}k' if x else '0')
FMT_PCT = mticker.PercentFormatter(1.0, decimals=0)


def linha_ref(ax, x, rotulo, y=0.97, cor=None):
    """Linha vertical de referência com rótulo no topo do eixo."""
    cor = cor or COR['texto2']
    ax.axvline(x, color=cor, lw=1, ls='--', zorder=3)
    ax.text(x, y, f' {rotulo}', transform=ax.get_xaxis_transform(), color=cor,
            fontsize=8, va='top', ha='left')


# ── Base da EDA: íntegras ligadas à divisão (são as que têm contagem de tokens)
df_eda = df[df['alvo'].notna()].copy()
df_eda['alvo'] = pd.Categorical(df_eda['alvo'], ORDEM_ALVO, ordered=True)
df_eda['chars_integra'] = df_eda['integra'].str.len()


def _estrutura_saida(txt):
    """Conta os elementos do JSON de saída (partes, temas e itens listados dentro dos temas)."""
    try:
        d = json.loads(txt)
    except (TypeError, ValueError):
        return {}
    if not isinstance(d, dict):
        return {}
    temas = [t for t in (d.get('Temas') or []) if isinstance(t, dict)]
    soma = lambda k: sum(len(t[k]) for t in temas if isinstance(t.get(k), list))
    return {'n_partes': len(d.get('Partes') or []), 'n_temas': len(temas),
            'n_argumentos': soma('Argumentos'), 'n_normas': soma('Normas'),
            'n_jurisprudencia': soma('Jurisprudencia'), 'n_doutrina': soma('Doutrina'),
            'n_conceitos': soma('Conceitos'),
            'n_preliminares': sum(str(t.get('Preliminar', '')).strip().lower().startswith('s') for t in temas)}


est = pd.DataFrame([_estrutura_saida(s) for s in df_eda['saida']], index=df_eda.index)
df_eda = df_eda.join(est)
print(f'base da EDA: {len(df_eda):,} íntegras com divisão | {df_eda["n_temas"].notna().sum():,} com saída JSON válida')

### Composição
Tamanho de cada conjunto e proporção de cada nível de dificuldade dentro dele. Faixas com a mesma largura nos três conjuntos indicam divisão estratificada.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={'width_ratios': [1, 1.6]})
fig.suptitle('Composição do dataset', x=0.01, ha='left')

# (a) instâncias por conjunto
cont = df_eda['alvo'].value_counts().reindex(ORDEM_ALVO)
ax1.barh(cont.index[::-1], cont.values[::-1], color=[COR_ALVO[a] for a in cont.index[::-1]], height=0.6)
for i, (a, v) in enumerate(cont[::-1].items()):
    ax1.text(v, i, f'  {v:,}  ({v / cont.sum():.1%})', va='center', color=COR['texto'], fontsize=9)
ax1.set_xlim(0, cont.max() * 1.35)
ax1.xaxis.set_major_formatter(FMT_MIL)
ax1.grid(axis='y', visible=False)
ax1.set_title(f'Instâncias por conjunto  (n = {cont.sum():,})')

# (b) proporção de cada nível de dificuldade dentro de cada conjunto (estratificação)
prop = pd.crosstab(df_eda['alvo'], df_eda['dificuldade_int'], normalize='index').reindex(ORDEM_ALVO[::-1])
esq = np.zeros(len(prop))
for j, nivel in enumerate(prop.columns):
    ax2.barh(prop.index.astype(str), prop[nivel], left=esq, color=RAMPA_DIF[j],
             edgecolor=COR['fundo'], linewidth=1.5, height=0.6)
    for i, v in enumerate(prop[nivel]):
        if v >= 0.06:
            ax2.text(esq[i] + v / 2, i, f'{v:.0%}', ha='center', va='center', fontsize=7.5,
                     color='white' if j >= 3 else COR['texto'])
    esq += prop[nivel].values
ax2.xaxis.set_major_formatter(FMT_PCT)
ax2.set_xlim(0, 1)
ax2.grid(axis='y', visible=False)
ax2.set_title('Distribuição de dificuldade por conjunto')
nomes_dif = df_eda.groupby('dificuldade_int')['dificuldade'].first()
ax2.legend([Patch(color=RAMPA_DIF[i]) for i in range(len(prop.columns))],
           [f'{n} ({nomes_dif[n]})' if n in (1, 4, 8) else str(n) for n in prop.columns],
           ncol=10, loc='upper center', bbox_to_anchor=(0.5, -0.12), fontsize=8,
           handlelength=1.2, columnspacing=1.0, title='dificuldade_int', title_fontsize=8)
plt.tight_layout()
plt.show()

### Distribuição de tokens
Histogramas de entrada (prompt + íntegra) e saída (JSON), com mediana, p90 e p99 marcados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
fig.suptitle('Distribuição da quantidade de tokens por instância', x=0.01, ha='left')

for ax, col in zip(axes, ['token_entrada', 'token_output']):
    s = df_eda[col].dropna().astype(float)
    ax.hist(s, bins=80, color=COR['azul'], edgecolor=COR['fundo'], linewidth=0.4)
    for q, rot in [(0.5, 'mediana'), (0.9, 'p90'), (0.99, 'p99')]:
        v = s.quantile(q)
        linha_ref(ax, v, f'{rot}\n {v:,.0f}', y=0.97 if q != 0.9 else 0.80)
    ax.set_title(f'{ROTULOS[col]}  ·  média {s.mean():,.0f} | máx {s.max():,.0f}')
    ax.set_xlabel('tokens')
    ax.set_ylabel('instâncias')
    ax.xaxis.set_major_formatter(FMT_K)
    ax.yaxis.set_major_formatter(FMT_MIL)
    ax.grid(axis='x', visible=False)

n_lim = (df_eda['token_output'] >= 8192).sum()
if n_lim:
    axes[1].annotate(f'{n_lim} saída(s) com 8.192 tokens\n(possível truncamento)',
                     xy=(8192, 0), xytext=(0.97, 0.45), textcoords='axes fraction', ha='right',
                     fontsize=8, color=COR['texto2'], arrowprops=dict(arrowstyle='-', color=COR['texto2'], lw=0.8))
plt.tight_layout()
plt.show()

### Cobertura por limite de contexto
Fração das instâncias que cabem em cada limite de tokens. Serve para escolher `max_seq_length` (total) e `max_new_tokens` (saída).

In [ ]:
# Fração das instâncias que cabem em cada limite de contexto (útil para definir max_seq_length do fine-tuning)
LIMITES_TOTAL = [16_384, 20_480, 24_576, 32_768]
LIMITES_SAIDA = [2_048, 4_096, 8_192]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
fig.suptitle('Cobertura por limite de tokens (distribuição acumulada)', x=0.01, ha='left')

for ax, col, limites in [(axes[0], 'token_total', LIMITES_TOTAL), (axes[1], 'token_output', LIMITES_SAIDA)]:
    s = np.sort(df_eda[col].dropna().astype(float).values)
    y = np.arange(1, len(s) + 1) / len(s)
    ax.plot(s, y, color=COR['azul'], lw=2)
    for i, lim in enumerate(limites):
        frac = (s <= lim).mean()
        fora = (s > lim).sum()
        ax.axvline(lim, color=COR['texto2'], lw=1, ls='--')
        ax.plot([lim], [frac], 'o', ms=6, color=COR['azul'], mec=COR['fundo'], mew=2, zorder=4)
        ax.text(lim, 0.08 + 0.17 * i, f' ≤ {lim:,}: {frac:.2%}\n {fora:,} acima', fontsize=8, color=COR['texto'])
    ax.set_title(f'{ROTULOS[col]}')
    ax.set_xlabel('tokens')
    ax.set_ylabel('fração acumulada de instâncias')
    ax.xaxis.set_major_formatter(FMT_K)
    ax.yaxis.set_major_formatter(FMT_PCT)
    ax.set_ylim(0, 1.02)
    ax.set_xlim(right=max(s.max(), limites[-1]) * 1.18)
plt.tight_layout()
plt.show()

print(f'{"limite (token_total)":<22}' + ''.join(f'{a:>12}' for a in ORDEM_ALVO) + f'{"todos":>12}')
for lim in LIMITES_TOTAL:
    linha = [(df_eda.loc[df_eda['alvo'] == a, 'token_total'] <= lim).mean() for a in ORDEM_ALVO]
    print(f'≤ {lim:<20,}' + ''.join(f'{v:>12.2%}' for v in linha) + f'{(df_eda["token_total"] <= lim).mean():>12.2%}')

### Tokens × dificuldade

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Tokens por nível de dificuldade  ·  caixa = p25–p75, bigodes = p5–p95, rótulo = mediana', x=0.01, ha='left')

niveis = sorted(df_eda['dificuldade_int'].dropna().unique())
for ax, col in zip(axes, ['token_entrada', 'token_output']):
    dados = [df_eda.loc[df_eda['dificuldade_int'] == n, col].dropna().astype(float) for n in niveis]
    bp = ax.boxplot(dados, positions=range(len(niveis)), widths=0.6, patch_artist=True,
                    showfliers=True, whis=(5, 95),
                    medianprops=dict(color='white', lw=1.6),
                    whiskerprops=dict(color=COR['texto2'], lw=1), capprops=dict(color=COR['texto2'], lw=1),
                    flierprops=dict(marker='.', ms=2, mfc=COR['texto2'], mec='none', alpha=0.35))
    for caixa, cor in zip(bp['boxes'], RAMPA_DIF):
        caixa.set(facecolor=cor, edgecolor=cor)
    for i, d in enumerate(dados):
        ax.text(i, d.quantile(.75), f'{d.median():,.0f}', ha='center', va='bottom', fontsize=7, color=COR['texto'])
    # separadores dos grupos fácil | médio | difícil
    for x in (2.5, 6.5):
        ax.axvline(x, color=COR['grade'], lw=1.2)
    for x, nome in [(1, 'fácil'), (4.5, 'médio'), (8, 'difícil')]:
        ax.text(x, 1.0, nome, transform=ax.get_xaxis_transform(), ha='center', va='bottom',
                fontsize=8.5, color=COR['texto2'])
    ax.set_xticks(range(len(niveis)), [str(n) for n in niveis])
    ax.set_xlabel('dificuldade_int')
    ax.set_ylabel('tokens')
    ax.yaxis.set_major_formatter(FMT_K)
    ax.grid(axis='x', visible=False)
    ax.set_title(ROTULOS[col], pad=16)
plt.tight_layout()
plt.show()

### Relações
No ajuste caracteres → tokens de entrada, o intercepto estima os tokens fixos do prompt e a inclinação dá a razão caracteres/token da íntegra.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), layout='constrained')
fig.suptitle('Relações entre tamanho da íntegra, entrada e saída', x=0.01, ha='left')

d = df_eda[['chars_integra', 'token_entrada', 'token_output']].dropna().astype(float)

# (a) caracteres da íntegra × tokens de entrada: o intercepto estima o custo fixo do prompt
inc, intercepto = np.polyfit(d['chars_integra'], d['token_entrada'], 1)
hb = ax1.hexbin(d['chars_integra'], d['token_entrada'], gridsize=60, bins='log', cmap=CMAP_AZUL, mincnt=1, linewidths=0)
xs = np.array([0, d['chars_integra'].max()])
ax1.plot(xs, intercepto + inc * xs, color=COR['laranja'], lw=1.6)
ax1.text(0.03, 0.95, f'ajuste linear: {intercepto:,.0f} + {inc * 1000:,.0f} tokens / 1.000 caracteres\n'
                     f'≈ {1 / inc:.2f} caracteres por token  ·  intercepto ≈ tokens fixos do prompt',
         transform=ax1.transAxes, va='top', fontsize=8, color=COR['texto'])
ax1.set_xlabel('caracteres da íntegra')
ax1.set_ylabel('tokens de entrada')
ax1.xaxis.set_major_formatter(FMT_K)
ax1.yaxis.set_major_formatter(FMT_K)
ax1.set_title('Íntegra (caracteres) × tokens de entrada')

# (b) entrada × saída
rho = d['token_entrada'].corr(d['token_output'], method='spearman')
hb2 = ax2.hexbin(d['token_entrada'], d['token_output'], gridsize=60, bins='log', cmap=CMAP_AZUL, mincnt=1, linewidths=0)
ax2.text(0.03, 0.95, f'Spearman ρ = {rho:.2f}', transform=ax2.transAxes, va='top', fontsize=8.5, color=COR['texto'])
ax2.set_xlabel('tokens de entrada')
ax2.set_ylabel('tokens de saída')
ax2.xaxis.set_major_formatter(FMT_K)
ax2.yaxis.set_major_formatter(FMT_K)
ax2.set_title('Tokens de entrada × tokens de saída')
for ax, h in [(ax1, hb), (ax2, hb2)]:
    cb = fig.colorbar(h, ax=ax, pad=0.01, fraction=0.04)
    cb.set_label('instâncias (escala log)')
    cb.outline.set_visible(False)
plt.show()

### Treino × validação × teste
Distribuições acumuladas sobrepostas. KS é a maior distância vertical em relação à curva do treino (0 = idênticas).

In [ ]:
# Os três conjuntos devem ter distribuições de tamanho equivalentes (curvas sobrepostas = divisão bem estratificada)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
fig.suptitle('Comparação entre treino, validação e teste', x=0.01, ha='left')

for ax, col in zip(axes, ['token_entrada', 'token_output']):
    ref = np.sort(df_eda.loc[df_eda['alvo'] == 'treino', col].dropna().astype(float).values)
    for alvo in ORDEM_ALVO:
        s = np.sort(df_eda.loc[df_eda['alvo'] == alvo, col].dropna().astype(float).values)
        y = np.arange(1, len(s) + 1) / len(s)
        # KS: maior distância vertical entre a curva do conjunto e a do treino
        grade = np.union1d(ref, s)
        ks = np.abs(np.searchsorted(ref, grade, side='right') / len(ref)
                    - np.searchsorted(s, grade, side='right') / len(s)).max()
        rot = f'{alvo}  (mediana {np.median(s):,.0f}' + (')' if alvo == 'treino' else f' · KS vs treino {ks:.3f})')
        ax.plot(s, y, color=COR_ALVO[alvo], lw=4 if alvo == 'treino' else 1.6, label=rot)
    ax.set_xlim(right=np.quantile(ref, 0.995))
    ax.set_title(f'{ROTULOS[col]}  (até p99,5)')
    ax.set_xlabel('tokens')
    ax.set_ylabel('fração acumulada')
    ax.xaxis.set_major_formatter(FMT_K)
    ax.yaxis.set_major_formatter(FMT_PCT)
    ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

### Metadados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), gridspec_kw={'height_ratios': [1, 1.15]})
fig.suptitle('Metadados dos acórdãos', x=0.01, ha='left')


def barras_h(ax, cont, titulo):
    ax.barh(cont.index.astype(str)[::-1], cont.values[::-1], color=COR['azul'], height=0.65)
    for i, v in enumerate(cont.values[::-1]):
        ax.text(v, i, f'  {v:,} ({v / len(df_eda):.1%})', va='center', fontsize=8, color=COR['texto'])
    ax.set_xlim(0, cont.max() * 1.3)
    ax.xaxis.set_major_formatter(FMT_MIL)
    ax.grid(axis='y', visible=False)
    ax.set_title(titulo)


# (a) classe processual
barras_h(axes[0, 0], df_eda['sg_classe'].value_counts(), 'Instâncias por classe processual (sg_classe)')

# (b) ramo do direito: tamanho da entrada por ramo, com n no rótulo
ax = axes[0, 1]
ramos = df_eda['sg_ramo_direito'].value_counts().index
dados = [df_eda.loc[df_eda['sg_ramo_direito'] == r, 'token_entrada'].dropna().astype(float) for r in ramos]
bp = ax.boxplot(dados[::-1], vert=False, widths=0.6, patch_artist=True, showfliers=False, whis=(5, 95),
                medianprops=dict(color='white', lw=1.6),
                whiskerprops=dict(color=COR['texto2']), capprops=dict(color=COR['texto2']))
for caixa in bp['boxes']:
    caixa.set(facecolor=COR['azul'], edgecolor=COR['azul'])
ax.set_yticks(range(1, len(ramos) + 1), [f'{r}  (n={len(d):,})' for r, d in zip(ramos[::-1], dados[::-1])])
ax.xaxis.set_major_formatter(FMT_K)
ax.grid(axis='y', visible=False)
ax.set_xlabel('tokens de entrada')
ax.set_title('Tokens de entrada por ramo do direito (sg_ramo_direito)')

# (c) linha do tempo de publicação
ax = axes[1, 0]
mes = pd.to_datetime(df_eda['dt_publicacao'], errors='coerce').dt.to_period('M')
cont_mes = mes.value_counts().sort_index()
cont_mes = cont_mes.reindex(pd.period_range(cont_mes.index.min(), cont_mes.index.max(), freq='M'), fill_value=0)
ax.bar(cont_mes.index.to_timestamp(), cont_mes.values, width=24, color=COR['azul'])
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 7)))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax.yaxis.set_major_formatter(FMT_MIL)
ax.grid(axis='x', visible=False)
ax.set_ylabel('instâncias')
ax.set_title('Acórdãos por mês de publicação (dt_publicacao)')
ano_cont = df_eda['ano'].value_counts().sort_index()
ax.text(0.02, 0.95, 'por ano: ' + '  ·  '.join(f'{a}: {v:,}' for a, v in ano_cont.items()),
        transform=ax.transAxes, va='top', fontsize=8, color=COR['texto2'])

# (d) relatores
ax = axes[1, 1]
cont_min = df_eda['num_ministro'].value_counts()
ax.bar(range(len(cont_min)), cont_min.values, color=COR['azul'], width=0.7)
ax.set_xticks(range(len(cont_min)), cont_min.index.astype(str), rotation=60, fontsize=7.5)
ax.yaxis.set_major_formatter(FMT_MIL)
ax.grid(axis='x', visible=False)
ax.set_xlabel('num_ministro')
ax.set_ylabel('instâncias')
top5 = cont_min.head(5).sum() / len(df_eda)
ax.set_title(f'Instâncias por ministro relator  ({len(cont_min)} ministros · 5 maiores = {top5:.0%})')
plt.tight_layout()
plt.show()

### Estrutura da saída esperada
Contagens extraídas do JSON de `saida`: temas, partes e itens listados dentro dos temas.

In [ ]:
d = df_eda[df_eda['n_temas'].notna()]
fig, axes = plt.subplots(2, 2, figsize=(13, 7.6))
fig.suptitle(f'Estrutura da saída esperada (JSON)  ·  {len(d):,} instâncias', x=0.01, ha='left')


def barras_discretas(ax, s, titulo, teto):
    """Histograma de contagem inteira; valores acima de 'teto' são agrupados em 'teto+'."""
    s = s.astype(int).clip(upper=teto)
    cont = s.value_counts().reindex(range(s.min(), teto + 1), fill_value=0)
    ax.bar(cont.index, cont.values, color=COR['azul'], width=0.7)
    for x, v in cont.items():
        if v / len(s) >= 0.01:
            ax.text(x, v, f'{v / len(s):.0%}', ha='center', va='bottom', fontsize=7.5, color=COR['texto'])
    ax.set_xticks(cont.index, [str(i) if i < teto else f'{teto}+' for i in cont.index])
    ax.yaxis.set_major_formatter(FMT_MIL)
    ax.grid(axis='x', visible=False)
    ax.set_ylabel('instâncias')
    ax.set_title(f'{titulo}  (média {s.mean():.1f})')


barras_discretas(axes[0, 0], d['n_temas'], 'Temas por acórdão', teto=8)
barras_discretas(axes[0, 1], d['n_partes'], 'Partes por acórdão', teto=6)

# (c) itens extraídos por acórdão (soma sobre os temas)
ax = axes[1, 0]
campos = {'n_argumentos': 'Argumentos', 'n_normas': 'Normas', 'n_jurisprudencia': 'Jurisprudência',
          'n_conceitos': 'Conceitos', 'n_doutrina': 'Doutrina'}
dados = [d[c].astype(float) for c in campos]
bp = ax.boxplot(dados, widths=0.6, patch_artist=True, showfliers=False, whis=(5, 95),
                medianprops=dict(color='white', lw=1.6),
                whiskerprops=dict(color=COR['texto2']), capprops=dict(color=COR['texto2']))
for caixa in bp['boxes']:
    caixa.set(facecolor=COR['azul'], edgecolor=COR['azul'])
ax.set_xticks(range(1, len(campos) + 1),
              [f'{n}\nmédia {s.mean():.1f}\nvazio em {(s == 0).mean():.0%}' for n, s in zip(campos.values(), dados)],
              fontsize=8)
ax.grid(axis='x', visible=False)
ax.set_ylabel('itens por acórdão')
ax.set_title('Itens extraídos por acórdão (soma dos temas; p5–p95)')

# (d) tamanho da saída cresce com o número de temas
ax = axes[1, 1]
temas = d['n_temas'].astype(int).clip(upper=8)
niveis = sorted(temas.unique())
dados = [d.loc[temas == n, 'token_output'].dropna().astype(float) for n in niveis]
bp = ax.boxplot(dados, positions=niveis, widths=0.6, patch_artist=True, showfliers=False, whis=(5, 95),
                medianprops=dict(color='white', lw=1.6),
                whiskerprops=dict(color=COR['texto2']), capprops=dict(color=COR['texto2']))
for caixa in bp['boxes']:
    caixa.set(facecolor=COR['azul'], edgecolor=COR['azul'])
ax.set_xticks(niveis, [f'{n}\n(n={len(x):,})' if n < 8 else f'8+\n(n={len(x):,})' for n, x in zip(niveis, dados)],
              fontsize=8)
ax.yaxis.set_major_formatter(FMT_K)
ax.grid(axis='x', visible=False)
ax.set_xlabel('temas no acórdão')
ax.set_ylabel('tokens de saída')
ax.set_title('Tokens de saída por número de temas')
plt.tight_layout()
plt.show()

prelim = d['n_preliminares'].gt(0).mean()
print(f'acórdãos com ao menos um tema preliminar: {prelim:.1%}')
print(f'temas por acórdão — mediana {d["n_temas"].median():.0f}, p90 {d["n_temas"].quantile(.9):.0f}, máx {d["n_temas"].max():.0f}')

## Exemplos de instâncias por conjunto

In [5]:
LARGURA = 110
MAX_CHARS_TEXTO = 1200   # trecho exibido de campos longos
LIMITE_CAMPO_LONGO = 150

def imprimir_instancia(row, titulo='', campos=None):
    campos = campos or list(row.index)
    larg_nome = max(len(c) for c in campos)
    print('═' * LARGURA)
    if titulo:
        print(f'  {titulo}')
        print('═' * LARGURA)
    for c in campos:
        v = row[c]
        if v is None or (not hasattr(v, '__len__') and pd.isna(v)):
            txt = '—'
        else:
            if c=='saida':
                try:
                   v = json.loads(v) if isinstance(v,str) else dict(v) 
                   txt = json.dumps(v) 
                except:
                   txt = str(v) 
            else:
                txt = str(v)
        n = len(txt)
        if n > LIMITE_CAMPO_LONGO and c != 'saida':
            corpo = ' '.join(txt.split())[:MAX_CHARS_TEXTO]
            corpo = textwrap.fill(corpo, LARGURA - 6, initial_indent='      ', subsequent_indent='      ')
            extra = '…' if n > MAX_CHARS_TEXTO else ''
            print(f'▸ {c}  ({n:,} caracteres)')
            print(corpo + extra)
        else:
            print(f'▸ {c:<{larg_nome}} : {txt}')
    print('─' * LARGURA + '\n')

def mostrar_exemplos(nome_alvo, n=2, seed=42):
    sub = df[df['alvo'] == nome_alvo]
    print(f'\n████ {nome_alvo.upper()} — {len(sub):,} instâncias; exibindo {min(n, len(sub))} ████\n')
    for i, (idx, row) in enumerate(sub.sample(min(n, len(sub)), random_state=seed).iterrows(), 1):
        imprimir_instancia(row, f'{nome_alvo.upper()} · exemplo {i} · índice {idx}')

print('Valores de alvo disponíveis:', list(df['alvo'].dropna().unique()))

Valores de alvo disponíveis: ['treino', 'teste', 'validacao']


In [6]:
# Treino
mostrar_exemplos('treino', n=2)


████ TREINO — 13,788 instâncias; exibindo 2 ████

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  TREINO · exemplo 1 · índice 12746
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
▸ sg_classe             : AREsp
▸ pasta                 : ./saidas/<<tipo>>_07
▸ fold_integras         : 7
▸ dt_publicacao         : 2024-12-19
▸ sg_ramo_direito       : CV
▸ num_registro          : 202403204748
▸ ano                   : 2024
▸ seq_documento_acordao : 286972747
▸ num_ministro          : 1143
▸ integra  (11,363 caracteres)
      ACÓRDÃO<br>Vistos e relatados estes autos em que são partes as acima indicadas, acordam os
      Ministros da QUARTA TURMA, por unanimidade, não conhecer do recurso, com aplicação de multa, nos
      termos do voto do Sr. Ministro Relator.<br>Os Srs. Ministros João Otávio de Noronha, Maria Isabel
      Gallotti, Antonio Carlos Ferreira e Marc

In [7]:
# Validação (ajuste o rótulo caso a lista acima mostre outro nome, ex.: 'validação')
mostrar_exemplos('validacao', n=2)


████ VALIDACAO — 1,976 instâncias; exibindo 2 ████

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  VALIDACAO · exemplo 1 · índice 18579
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
▸ sg_classe             : AREsp
▸ pasta                 : ./saidas/<<tipo>>_10
▸ fold_integras         : 10
▸ dt_publicacao         : 2024-08-16
▸ sg_ramo_direito       : PN
▸ num_registro          : 202401170054
▸ ano                   : 2024
▸ seq_documento_acordao : 263620365
▸ num_ministro          : 1184
▸ integra  (6,426 caracteres)
      ACÓRDÃO<br>Vistos e relatados estes autos em que são partes as acima indicadas, acordam os
      Ministros da Turma, por unanimidade, negar provimento ao agravo regimental.<br>Os Srs. Ministros
      Reynaldo Soares da Fonseca, Ribeiro Dantas, Joel Ilan Paciornik e Daniela Teixeira votaram com o
      Sr. Ministro Relator.<br>Presidiu o 

In [8]:
# Teste
mostrar_exemplos('teste', n=2)


████ TESTE — 3,948 instâncias; exibindo 2 ████

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  TESTE · exemplo 1 · índice 15945
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
▸ sg_classe             : AREsp
▸ pasta                 : ./saidas/<<tipo>>_06
▸ fold_integras         : 6
▸ dt_publicacao         : 2024-11-13
▸ sg_ramo_direito       : CV
▸ num_registro          : 202402786661
▸ ano                   : 2024
▸ seq_documento_acordao : 281725292
▸ num_ministro          : 1123
▸ integra  (5,285 caracteres)
      ACÓRDÃO<br>Vistos e relatados estes autos em que são partes as acima indicadas, acordam os
      Ministros da QUARTA TURMA do Superior Tribunal de Justiça, em sessão virtual de 05/11/2024 a
      11/11/2024, por unanimidade, negar provimento ao recurso, nos termos do voto do Sr. Ministro
      Relator.<br>Os Srs. Ministros Raul Araújo, Maria Isab